# Advanced PySpark Optimization — Examples 21–30

Notebook 3 of the 5-notebook Advanced PySpark Optimization series.

## Examples

21. Cache vs. no cache
22. `cache()` vs. `persist()`
23. `MEMORY_AND_DISK` and storage levels
24. Cache lifecycle, eviction concepts, and `unpersist()`
25. Shuffle spill investigation and observability
26. Spark memory configuration inspection
27. When caching can make a workload slower
28. Default join strategy and physical-plan inspection
29. Broadcast join optimization
30. Auto broadcast threshold and join strategy changes

## Important lab note

The CSV files intentionally contain only 10 rows. For that reason, caching, spilling, and eviction are **not expected to produce production-like timings** here.

These examples focus on:

- mechanics
- storage levels
- execution plans
- observability
- correct optimization reasoning

Where a phenomenon such as memory eviction or shuffle spill cannot be reliably forced on a tiny local dataset, the notebook explicitly says so rather than pretending that it happened.

## Source mapping

This notebook follows the source material's memory, persistence, shuffle, and join-optimization sections, including:

- cache and persistence
- storage levels
- memory vs. disk persistence
- cache lifecycle and memory pressure concepts
- shuffle spill concepts and observability
- executor/driver memory configuration awareness
- unnecessary caching
- default join strategies
- broadcast joins
- broadcast thresholds

The examples intentionally combine related pointers into one lab when that produces a more realistic optimization workflow.

In [ ]:
from pathlib import Path
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.storagelevel import StorageLevel

BASE_PATH = Path.cwd()
DATA_PATH = BASE_PATH / "data"

if not DATA_PATH.exists():
    candidate = Path("/mnt/data/pyspark_examples_21_30/data")
    if candidate.exists():
        DATA_PATH = candidate

assert DATA_PATH.exists(), "Could not find data/. Update BASE_PATH."

spark = (
    SparkSession.builder
    .appName("Advanced-PySpark-Examples-21-30")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
])

orders_df = (
    spark.read.option("header", True)
    .schema(orders_schema)
    .csv(str(DATA_PATH / "orders.csv"))
)

customers_df = (
    spark.read.option("header", True)
    .schema(customers_schema)
    .csv(str(DATA_PATH / "customers.csv"))
)

print("Spark version:", spark.version)
print("Orders:", orders_df.count())
print("Customers:", customers_df.count())

# Example 21 — Cache vs. No Cache

**Concepts:** repeated computation, lazy cache materialization, cache usefulness.

Caching is most useful when the same expensive lineage is reused by multiple downstream actions.

We deliberately create a reusable transformation and run multiple actions with and without caching.

**Do not treat timing from this tiny dataset as a benchmark.** On only a few rows, cache overhead can be as large as or larger than recomputation.

In [ ]:
def build_reusable_df():
    return (
        orders_df
        .filter(F.col("amount") >= 300)
        .withColumn("discounted_amount", F.col("amount") * F.lit(0.9))
        .withColumn("amount_band",
                    F.when(F.col("amount") >= 1000, "HIGH")
                     .when(F.col("amount") >= 500, "MEDIUM")
                     .otherwise("LOW"))
    )

# Without cache
no_cache_df = build_reusable_df()

start = time.perf_counter()
no_cache_count = no_cache_df.count()
no_cache_sum = no_cache_df.agg(F.sum("discounted_amount")).first()[0]
no_cache_seconds = time.perf_counter() - start

# With cache
cached_df = build_reusable_df().cache()

print("Is cached before first action?", cached_df.is_cached)

start = time.perf_counter()
cached_count = cached_df.count()  # materializes the cache
cached_sum = cached_df.agg(F.sum("discounted_amount")).first()[0]
cached_seconds = time.perf_counter() - start

print("Is cached after action?", cached_df.is_cached)
print("No-cache count/sum:", no_cache_count, no_cache_sum)
print("Cached count/sum:", cached_count, cached_sum)
print("No-cache elapsed:", round(no_cache_seconds, 6), "seconds")
print("Cached elapsed:", round(cached_seconds, 6), "seconds")

assert no_cache_count == cached_count
assert no_cache_sum == cached_sum

cached_df.unpersist()

# Example 22 — `cache()` vs. `persist()`

**Concepts:** default storage level, explicit storage levels, persistence choices.

`cache()` is a convenience API. `persist()` lets you choose a storage level explicitly.

The key optimization question is not simply "should I cache?" but:

> How many times is this lineage reused, and what storage trade-off is appropriate?

Inspect `storageLevel` before and after persistence.

In [ ]:
base_df = (
    orders_df
    .withColumn("amount_x2", F.col("amount") * 2)
    .filter(F.col("amount_x2") > 500)
)

cached = base_df.cache()
print("cache() storage level:", cached.storageLevel)
cached.count()  # materialize

persisted = base_df.persist(StorageLevel.MEMORY_AND_DISK)
print("persist(MEMORY_AND_DISK) storage level:", persisted.storageLevel)
persisted.count()  # materialize

print("\nCached rows:")
cached.show()

print("\nPersisted rows:")
persisted.show()

cached.unpersist()
persisted.unpersist()

# Example 23 — `MEMORY_AND_DISK` and Storage Levels

**Concepts:** storage memory, disk fallback, persistence trade-offs.

`MEMORY_AND_DISK` allows cached partitions to spill to disk if they cannot all remain in memory.

With this tiny dataset, Spark will usually keep everything in memory, so this example demonstrates the configured behavior rather than forcing disk fallback.

In [ ]:
memory_and_disk_df = (
    orders_df
    .filter(F.col("amount") > 100)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Configured storage level:", memory_and_disk_df.storageLevel)
print("Cached before action?", memory_and_disk_df.is_cached)

memory_and_disk_df.count()

print("Cached after materialization?", memory_and_disk_df.is_cached)

# Run another action that can reuse the persisted lineage.
memory_and_disk_df.groupBy("category").agg(
    F.sum("amount").alias("total_amount")
).show()

memory_and_disk_df.unpersist()
print("Cached after unpersist()?", memory_and_disk_df.is_cached)

# Example 24 — Cache Lifecycle, Eviction Concepts, and `unpersist()`

**Concepts:** cache lifecycle, memory pressure concepts, cache cleanup, eviction awareness.

A tiny local dataset cannot reliably force Spark to evict cached partitions. Instead, this lab demonstrates the lifecycle you can directly control:

1. create reusable DataFrames
2. materialize persistence
3. inspect cache state
4. release storage with `unpersist()`

In production, eviction can occur when storage memory competes with other workloads. Do not rely on cache survival indefinitely.

In [ ]:
df_a = orders_df.filter(F.col("category") == "Electronics").persist()
df_b = orders_df.filter(F.col("category") == "Furniture").persist()

print("Before actions:")
print("df_a cached:", df_a.is_cached)
print("df_b cached:", df_b.is_cached)

df_a.count()
df_b.count()

print("\nAfter materialization:")
print("df_a cached:", df_a.is_cached, "| storage:", df_a.storageLevel)
print("df_b cached:", df_b.is_cached, "| storage:", df_b.storageLevel)

print("\nReleasing df_a explicitly...")
df_a.unpersist()
print("df_a cached:", df_a.is_cached)
print("df_b cached:", df_b.is_cached)

print("\nReleasing df_b explicitly...")
df_b.unpersist()
print("df_b cached:", df_b.is_cached)

# Example 25 — Shuffle Spill Investigation and Observability

**Concepts:** execution memory, memory spill, disk spill, shuffle metrics.

A 10-row dataset is not large enough to reliably trigger shuffle spill. Therefore this example creates a shuffle-producing aggregation and shows:

- where the shuffle appears in the plan
- which Spark UI metrics to inspect when running a larger version

For a real spill investigation, run the same pattern on sufficiently large data and inspect stage/task metrics such as:

- memory bytes spilled
- disk bytes spilled
- shuffle read/write
- peak execution memory
- task duration imbalance

In [ ]:
shuffle_df = (
    orders_df
    .repartition(4, "category")
    .groupBy("category")
    .agg(
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.count("*").alias("order_count")
    )
)

print("PHYSICAL PLAN — LOOK FOR EXCHANGE / SHUFFLE")
shuffle_df.explain("formatted")

print("\nExecuting aggregation:")
shuffle_df.show()

print("\nLab note:")
print("No spill is expected from 10 rows.")
print("Use the Spark UI on a larger workload to inspect memory and disk spill metrics.")

# Example 26 — Spark Memory Configuration Inspection

**Concepts:** driver memory, executor memory, memory overhead, configuration awareness.

Many Spark memory settings are determined when the Spark application starts. Changing them inside a running session may not reconfigure the already-created JVM.

This example therefore inspects the active configuration and explains which values are launch-time decisions.

In [ ]:
memory_related_keys = [
    "spark.driver.memory",
    "spark.executor.memory",
    "spark.executor.memoryOverhead",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
    "spark.sql.shuffle.partitions",
]

print("Active / visible memory-related configuration:")
for key in memory_related_keys:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key} = <not explicitly set in this Spark session>")

print("\nSparkContext defaultParallelism:", spark.sparkContext.defaultParallelism)

print("\nImportant:")
print("- driver/executor memory is typically configured before application startup")
print("- execution and storage workloads compete for available executor resources")
print("- changing memory values without understanding the workload can make performance worse")

# Example 27 — When Caching Can Make a Workload Slower

**Concepts:** unnecessary caching, cache materialization overhead, workload-dependent optimization.

We compare a single-use computation with and without caching.

For one downstream action, caching may add overhead because Spark must both compute the result and store it.

On this tiny dataset, timings are intentionally illustrative only.

In [ ]:
def single_use_pipeline():
    return (
        orders_df
        .filter(F.col("amount") > 400)
        .withColumn("taxed_amount", F.col("amount") * F.lit(1.1))
        .groupBy("category")
        .agg(F.sum("taxed_amount").alias("total_taxed_amount"))
    )

# Single action without cache
uncached_single_use = single_use_pipeline()
start = time.perf_counter()
uncached_result = uncached_single_use.collect()
uncached_time = time.perf_counter() - start

# Single action with cache
cached_single_use = single_use_pipeline().cache()
start = time.perf_counter()
cached_result = cached_single_use.collect()
cached_time = time.perf_counter() - start

cached_single_use.unpersist()

print("Uncached single-use elapsed:", round(uncached_time, 6), "seconds")
print("Cached single-use elapsed:", round(cached_time, 6), "seconds")
print("Results identical:", sorted(uncached_result) == sorted(cached_result))

print("\nOptimization lesson:")
print("Caching should be justified by reuse, not added automatically.")

# Example 28 — Default Join Strategy and Physical-Plan Inspection

**Concepts:** join planning, physical plans, sort-merge/broadcast possibilities, shuffle.

Spark's chosen join strategy depends on configuration, statistics, data sizes, and available optimization information.

This example does **not** force a strategy. It lets Spark choose and then inspects the plan.

In [ ]:
default_join_df = (
    orders_df
    .join(customers_df, on="customer_id", how="inner")
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "segment",
        "product",
        "category",
        "amount"
    )
)

print("DEFAULT JOIN PLAN")
default_join_df.explain("formatted")

print("\nDEFAULT JOIN RESULT")
default_join_df.orderBy("order_id").show()

# Example 29 — Broadcast Join Optimization

**Concepts:** broadcast join, avoiding redistribution of a small dimension table.

Here we explicitly broadcast the small `customers_df`.

Inspect the plan and look for a broadcast-related operator such as `BroadcastExchange` / `BroadcastHashJoin`.

In [ ]:
broadcast_join_df = (
    orders_df
    .join(F.broadcast(customers_df), on="customer_id", how="inner")
    .select(
        "order_id",
        "customer_id",
        "customer_name",
        "segment",
        "product",
        "category",
        "amount"
    )
)

print("BROADCAST JOIN PLAN")
broadcast_join_df.explain("formatted")

print("\nBROADCAST JOIN RESULT")
broadcast_join_df.orderBy("order_id").show()

assert default_join_df.count() == broadcast_join_df.count()
print("Row-count validation passed.")

# Example 30 — Auto Broadcast Threshold and Join Strategy Changes

**Concepts:** `spark.sql.autoBroadcastJoinThreshold`, automatic join selection, configuration trade-offs.

We compare:

1. automatic planning with the current threshold
2. automatic broadcast disabled by setting the threshold to `-1`
3. explicit broadcast hint

The exact plan can vary by Spark version and local configuration, so inspect the output rather than assuming a specific operator name.

In [ ]:
original_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

def explain_auto_join(label):
    join_df = orders_df.join(customers_df, "customer_id")
    print(f"\n--- {label} ---")
    print("Threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
    join_df.explain("formatted")
    return join_df

try:
    # Current/default behavior
    current_join = explain_auto_join("CURRENT AUTO-BROADCAST CONFIGURATION")

    # Disable automatic broadcast.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    no_auto_broadcast_join = explain_auto_join("AUTO-BROADCAST DISABLED")

    # Explicit broadcast remains an intentional hint.
    explicit_broadcast_join = (
        orders_df
        .join(F.broadcast(customers_df), "customer_id")
    )

    print("\n--- EXPLICIT BROADCAST ---")
    explicit_broadcast_join.explain("formatted")

    assert current_join.count() == no_auto_broadcast_join.count()
    assert current_join.count() == explicit_broadcast_join.count()
    print("\nAll join variants produced the same row count.")

finally:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", original_threshold)

print("\nRestored original threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Notebook 3 Summary

Examples 21–30 focused on two major optimization areas.

## Memory and persistence

- lazy cache materialization
- `cache()` vs. `persist()`
- storage levels
- `MEMORY_AND_DISK`
- explicit `unpersist()`
- cache lifecycle
- memory-pressure concepts
- shuffle spill observability
- active Spark memory configuration
- why caching is not automatically beneficial

## Join optimization

- inspect the default physical plan
- understand that join strategy is chosen from runtime/planning information
- explicitly broadcast a small table
- inspect `spark.sql.autoBroadcastJoinThreshold`
- compare automatic and explicit join planning

## Core optimization rule

Do not optimize based only on API names.

Instead:

1. inspect the plan
2. understand data size and reuse patterns
3. apply one optimization
4. validate correctness
5. inspect the new plan and runtime metrics

## Next notebook

Examples 31–40 will move into:

- broadcast join trade-offs
- sort-merge joins
- bucket-aware joins
- repartitioning before joins
- join skew detection
- skew mitigation patterns
- Catalyst optimization walkthrough
- predicate pushdown
- partition pruning vs. predicate pushdown
- built-in functions vs. Python UDFs